# Path Matters — Results Exploration

This notebook loads `results/metrics/reconstruction_results.csv` and generates
all publication figures:

1. Model comparison (VGGT vs Fast3R vs SAM3D)
2. Trajectory × orientation heatmaps (fitness and RMSE)
3. ICP vs BUFFER-X comparison
4. RL experiment comparison (exp_06 vs exp_07)
5. Scale estimation error distribution

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
from pathlib import Path

# Dark-navy publication style used throughout the paper
BG    = '#0D1B2A'
CYAN  = '#00B4D8'
AMBER = '#FF6D00'
BLUE  = '#1565C0'
GRID  = '#1D3461'

matplotlib.rcParams.update({
    'figure.facecolor': BG,
    'axes.facecolor':   BG,
    'axes.edgecolor':   GRID,
    'axes.labelcolor':  'white',
    'xtick.color':      'gray',
    'ytick.color':      'gray',
    'text.color':       'white',
    'grid.color':       GRID,
    'grid.linestyle':   '--',
    'font.size':        10,
})

RESULTS_CSV = Path('../results/metrics/reconstruction_results.csv')
FIGURES_DIR = Path('../results/figures')
FIGURES_DIR.mkdir(exist_ok=True)

df = pd.read_csv(RESULTS_CSV)
print(f'Loaded {len(df)} rows from {RESULTS_CSV}')
df.head()

## 1. Model Comparison (Table 7)

In [ ]:
table7 = df[df['experiment'].str.startswith('Table7')].copy()
table7['model_label'] = table7['model']

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle('Table 7: Reconstruction Model Comparison (28 objects)', color='white', y=1.02)

metrics = [
    ('fitness_mean', 'Alignment Fitness (↑)', [0.5, 1.0]),
    ('rmse_mean_m',  'Inlier RMSE (m) (↓)',   [0.0, 0.015]),
    ('density_pts',  'Point Density (pts) (↑)', [0, 50000]),
]
colours_model = [CYAN, AMBER, BLUE]

for ax, (col, label, ylim) in zip(axes, metrics):
    bars = ax.bar(table7['model_label'], table7[col], color=colours_model, width=0.5)
    ax.set_ylabel(label)
    ax.set_ylim(ylim)
    ax.set_title(label.split(' (')[0])
    ax.grid(axis='y', alpha=0.4)
    for bar, val in zip(bars, table7[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (ylim[1]-ylim[0])*0.02,
                f'{val:.3f}' if val < 1 else f'{int(val):,}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
out = FIGURES_DIR / 'fig_model_comparison.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Saved: {out}')

## 2. Trajectory × Camera Orientation (Table 11)

In [ ]:
table11 = df[df['experiment'].str.startswith('Table11')].copy()
table11['approach'] = table11['camera_approach']

trajectories = ['Lawnmower', 'Zigzag', 'Hemisphere', 'Spiral', 'Random']
approaches   = ['Approach1', 'Approach2']

for metric, ylabel, title_suffix in [
    ('fitness_mean', 'ICP Fitness (↑)',    'Fitness'),
    ('rmse_mean_m',  'Inlier RMSE (m) (↓)', 'RMSE'),
]:
    fig, ax = plt.subplots(figsize=(9, 4))
    x    = np.arange(len(trajectories))
    w    = 0.35
    cols = [CYAN, AMBER]

    for i, (approach, col) in enumerate(zip(approaches, cols)):
        subset = table11[table11['approach'] == approach]
        vals   = [subset[subset['trajectory'] == t][metric].values[0]
                  if len(subset[subset['trajectory'] == t]) > 0 else 0
                  for t in trajectories]
        bars = ax.bar(x + i * w, vals, w, label=approach.replace('Approach', 'Approach '),
                      color=col, alpha=0.85)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                    f'{val:.3f}' if val < 1 else f'{val:.4f}',
                    ha='center', va='bottom', fontsize=8)

    ax.set_xticks(x + w / 2)
    ax.set_xticklabels(trajectories)
    ax.set_ylabel(ylabel)
    ax.set_title(f'Table 11: {title_suffix} by Trajectory & Camera Orientation')
    ax.legend()
    ax.grid(axis='y', alpha=0.4)
    plt.tight_layout()
    out = FIGURES_DIR / f'fig_{metric.split("_")[0]}_by_trajectory.png'
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.show()
    print(f'Saved: {out}')

## 3. ICP vs BUFFER-X Comparison

In [ ]:
reg = df[df['experiment'].str.startswith('ICP_vs_BUFFERX')].copy()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
fig.suptitle('ICP vs BUFFER-X Registration', color='white')

for ax, (col, label) in zip(axes, [('fitness_mean', 'Fitness (↑)'), ('rmse_mean_m', 'RMSE m (↓)')]):
    bars = ax.bar(reg['notes'].str.split(' registration').str[0],
                  reg[col], color=[CYAN, AMBER], width=0.5)
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.grid(axis='y', alpha=0.4)
    for bar, val in zip(bars, reg[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.3f}', ha='center', va='bottom')

plt.tight_layout()
out = FIGURES_DIR / 'fig_icp_vs_bufferx.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Saved: {out}')

## 4. RL Experiment Comparison (exp_06 vs exp_07)

In [ ]:
rl_data = {
    'Experiment':    ['exp_06\n(no shaping)', 'exp_07\n(proximity shaping)', 'Random\nbaseline'],
    'Success Rate':  [0.004, 0.452, 0.0],
    'Mean Reward':   [22.5, 32.7, 15.0],
    'Coverage':      [0.20, 0.75, 0.206],
}

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle('RL Viewpoint Selection: Experiment Comparison', color='white')

for ax, (key, ylabel) in zip(axes, [
    ('Success Rate', 'Episode Success Rate'),
    ('Mean Reward',  'Mean Episode Reward'),
    ('Coverage',     'Workspace Coverage Ratio'),
]):
    bars = ax.bar(rl_data['Experiment'], rl_data[key],
                  color=[BLUE, CYAN, AMBER], width=0.6)
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.grid(axis='y', alpha=0.4)
    for bar, val in zip(bars, rl_data[key]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(rl_data[key])*0.02,
                f'{val:.1%}' if val <= 1 else f'{val:.1f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
out = FIGURES_DIR / 'fig_rl_experiments.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Saved: {out}')

## 5. Summary Statistics

In [ ]:
print('=== Model Comparison (Table 7) ===')
print(df[df['experiment'].str.startswith('Table7')]
      [['model', 'fitness_mean', 'rmse_mean_m', 'density_pts', 'time_s']]
      .to_string(index=False))

print('\n=== Best fixed strategy ===')
best = df.loc[df['fitness_mean'].idxmax()]
print(f"  {best['experiment']}: fitness={best['fitness_mean']:.3f}, RMSE={best['rmse_mean_m']:.4f}m")

print('\n=== Camera orientation impact ===')
a1 = df[df['camera_approach']=='Approach1']['fitness_mean'].mean()
a2 = df[df['camera_approach']=='Approach2']['fitness_mean'].mean()
print(f'  Approach 1 (fixed down):     mean fitness = {a1:.3f}')
print(f'  Approach 2 (object-pointing): mean fitness = {a2:.3f}')
print(f'  Improvement: +{(a2-a1)/a1*100:.1f}%')